![CCAI 9012 course banner](../../../docs/figs/brand_identity/banner.png)

# Credit audit: can a model be accurate and still treat age groups differently?

A bank trains a classifier on past credit outcomes. We compare approval patterns for applicants below 25 and those 25 or older, then change training weights and ask what happens to both prediction quality and group differences. This is an educational audit, not a lending rule.

## What you will learn

- Read a protected-group definition and a favourable outcome in the data.
- Explain what reweighing changes in the training sample.
- Compare accuracy and AUC on a shared held-out set.
- Read signed fairness differences and identify the favoured group.

![Credit records branch into reweighted training, predictions, performance and signed fairness checks.](../../../docs/figs/credit_teaching_flow.svg)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from copy import deepcopy
from IPython.display import display
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import BinaryLabelDatasetMetric, ClassificationMetric
from aif360.algorithms.preprocessing import Reweighing
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from ccai9012.credit_utils import load_german_credit_dataset
from ccai9012.paths import DATA_DIR


## Step 1 — Load the tracked credit records

The repository stores `data/german_credits/german.data`. The helper reads it directly, encodes age as under 25 versus 25 or older, and defines good credit as the favourable label. The split uses one fixed seed for repeatability.


In [ ]:
dataset_orig = load_german_credit_dataset(DATA_DIR / "german_credits/german.data")
dataset_orig_train, dataset_orig_test = dataset_orig.split([0.7], shuffle=True, seed=42)
privileged_groups = [{"age": 1}]
unprivileged_groups = [{"age": 0}]
print(f"train={len(dataset_orig_train.labels)}; test={len(dataset_orig_test.labels)}")


## Step 2 — Read the training outcome gap

`mean_difference` is the favourable outcome rate for under-25 applicants minus the rate for applicants 25 or older. A negative number favours the older group in this sample. Read the printed value before judging its size.


In [ ]:
metric_orig_train = BinaryLabelDatasetMetric(dataset_orig_train, unprivileged_groups=unprivileged_groups, privileged_groups=privileged_groups)
original_gap = metric_orig_train.mean_difference()
print(f"Training outcome gap (under 25 − age 25+): {original_gap:+.3f}")


## Step 3 — Reweigh the training rows

Reweighing changes each row’s training weight based on its age group and outcome. It does not edit the held-out labels. Compare the signed gap after weighting; closeness to zero on training data does not guarantee fair predictions.


In [ ]:
RW = Reweighing(unprivileged_groups=unprivileged_groups,
                privileged_groups=privileged_groups)
dataset_transf_train = RW.fit_transform(dataset_orig_train)

## Step 4 — Check the weighted training gap

The sign keeps its meaning: under-25 favourable rate minus age-25+ favourable rate. The new value comes from this run, not a fixed expected percentage.


In [ ]:
metric_transf_train = BinaryLabelDatasetMetric(dataset_transf_train, unprivileged_groups=unprivileged_groups, privileged_groups=privileged_groups)
weighted_gap = metric_transf_train.mean_difference()
print(f"Weighted training outcome gap (under 25 − age 25+): {weighted_gap:+.3f}")


## Step 5: How is the data transformed?

The Reweighing algorithm is a preprocessing bias mitigation technique that assigns weights to training samples based on their protected group membership and class label. 

- It calculates the expected distribution of favorable and unfavorable labels across privileged and unprivileged groups.
- Then, it adjusts the weight of each sample to correct imbalances, effectively balancing the joint distribution of `(protected attribute, label)` pairs.
- For example, if an unprivileged group has fewer favorable outcomes, samples from that group with favorable labels will receive higher weights.
- Conversely, samples from privileged groups or more frequent `(group, label)` combinations will have their weights decreased.
  
This weighting scheme allows the model to "see" a balanced dataset during training, promoting fairness by reducing the bias towards privileged groups.

The following visualization illustrates how sample weights are distributed before and after Reweighing, showing that originally all samples have equal weight (1.0), while after transformation weights vary depending

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Extract protected attribute, label, and weights from a BinaryLabelDataset
def extract_weights(dataset):
    # If instance_weights do not exist (e.g., original data), assume weight = 1 for all
    weights = dataset.instance_weights if hasattr(dataset, 'instance_weights') else np.ones(len(dataset.labels))
    df = pd.DataFrame({
        'protected': dataset.protected_attributes[:, 0],  # e.g., age binary attribute
        'label': dataset.labels[:, 0],                    # class label
        'weight': weights
    })
    return df

# Extract data for original and reweighted datasets
df_orig = extract_weights(dataset_orig_train)
df_rw = extract_weights(dataset_transf_train)

protected_mapping = {
    0: "Unprivileged (Age < 25)",
    1: "Privileged (Age ≥ 25)"
}
label_mapping = {
    0: "Unfavorable (Bad Credit)",
    1: "Favorable (Good Credit)"
}

df_orig['protected_str'] = df_orig['protected'].map(protected_mapping)
df_orig['label_str'] = df_orig['label'].map(label_mapping)

df_rw['protected_str'] = df_rw['protected'].map(protected_mapping)
df_rw['label_str'] = df_rw['label'].map(label_mapping)

# Compute mean weights per (protected_str, label_str) group
grouped_orig = df_orig.groupby(['protected_str', 'label_str'])['weight'].mean().reset_index()
grouped_rw = df_rw.groupby(['protected_str', 'label_str'])['weight'].mean().reset_index()

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

sns.barplot(data=grouped_orig, x='label_str', y='weight', hue='protected_str', ax=axes[0])
axes[0].set_title('Original Training Data: Sample Weights')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Mean Sample Weight')
axes[0].legend(title='Protected Attribute')

sns.barplot(data=grouped_rw, x='label_str', y='weight', hue='protected_str', ax=axes[1])
axes[1].set_title('Reweighed Training Data: Sample Weights')
axes[1].set_xlabel('Label')
axes[1].legend(title='Protected Attribute')

plt.tight_layout()
plt.show()

In [ ]:
print("Original grouped weights:")
print(grouped_orig)

print("\nReweighted grouped weights:")
print(grouped_rw)

## Step 6: Train 2 models with original and transformed datasets

In [ ]:
# Model imports are collected in the setup cell.


In [ ]:
# Helper function to extract features (X), labels (y), and sample weights (w) from a BinaryLabelDataset
def dataset_to_X_y_w(dataset: BinaryLabelDataset):
    X = dataset.features                    # feature matrix (numpy array)
    y = dataset.labels.ravel()              # labels as 1D array
    # if instance weights exist (e.g., after reweighing), use them; otherwise assume all weights = 1
    w = dataset.instance_weights if hasattr(dataset, 'instance_weights') else np.ones(len(y))
    return X, y, w

In [ ]:
# Convert to X,y,w for original training data
X_train_orig, y_train_orig, w_train_orig = dataset_to_X_y_w(dataset_orig_train)

# Convert to X,y,w for reweighed training data
X_train_rw, y_train_rw, w_train_rw = dataset_to_X_y_w(dataset_transf_train)

# Convert to X,y,w for test data
X_test, y_test, w_test = dataset_to_X_y_w(dataset_orig_test)

In [ ]:
# Train model on original training data
model_orig = LGBMClassifier(random_state=42)
model_orig.fit(X_train_orig, y_train_orig, sample_weight=w_train_orig)
y_pred_orig = model_orig.predict(X_test)
y_prob_orig = model_orig.predict_proba(X_test)[:, 1]

# Train model on reweighed training data
model_rw = LGBMClassifier(random_state=42)
model_rw.fit(X_train_rw, y_train_rw, sample_weight=w_train_rw)
y_pred_rw = model_rw.predict(X_test)
y_prob_rw = model_rw.predict_proba(X_test)[:, 1]

## Step 6 — Compare two models on the same test records

Accuracy and AUC describe prediction performance. Statistical parity and equal opportunity describe different group gaps. Neither pair of values should be combined into a single score.


In [ ]:
# Prepare predicted label datasets for fairness metric calculation
dataset_orig_test_pred_orig = deepcopy(dataset_orig_test)
dataset_orig_test_pred_orig.labels = y_pred_orig.reshape(-1, 1)

dataset_orig_test_pred_rw = deepcopy(dataset_orig_test)
dataset_orig_test_pred_rw.labels = y_pred_rw.reshape(-1, 1)

In [ ]:
acc_orig = accuracy_score(y_test, y_pred_orig)
auc_orig = roc_auc_score(y_test, y_prob_orig)
acc_rw = accuracy_score(y_test, y_pred_rw)
auc_rw = roc_auc_score(y_test, y_prob_rw)
metric_orig = ClassificationMetric(dataset_orig_test, dataset_orig_test_pred_orig, unprivileged_groups=unprivileged_groups, privileged_groups=privileged_groups)
metric_rw = ClassificationMetric(dataset_orig_test, dataset_orig_test_pred_rw, unprivileged_groups=unprivileged_groups, privileged_groups=privileged_groups)
performance = pd.DataFrame({"model": ["original", "reweighed"], "accuracy": [acc_orig, acc_rw], "auc": [auc_orig, auc_rw]})
fairness = pd.DataFrame({"model": ["original", "reweighed"], "statistical_parity_difference": [metric_orig.statistical_parity_difference(), metric_rw.statistical_parity_difference()], "equal_opportunity_difference": [metric_orig.equal_opportunity_difference(), metric_rw.equal_opportunity_difference()]})
print("Prediction performance")
display(performance)
print("Signed fairness gaps: under 25 minus age 25+; negative values favour the older group")
display(fairness)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
performance.set_index("model")[["accuracy", "auc"]].plot.bar(ax=axes[0], ylim=(0, 1), title="Held-out prediction performance")
fairness.set_index("model")[["statistical_parity_difference", "equal_opportunity_difference"]].plot.bar(ax=axes[1], title="Signed fairness differences")
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_ylabel("Under 25 minus age 25+")
plt.tight_layout()
plt.show()


### Read the results together

First compare accuracy and AUC. Then inspect each signed fairness difference: below zero means the under-25 group has a lower favourable prediction rate (or lower true-positive rate for equal opportunity). Ask whether reweighing moved each metric toward zero and whether predictive performance changed. This one split is not proof of legal fairness or deployment readiness.
